# 09: Nerfstudio Integration for Aria Reconstruction

In this notebook we will demonstrate a reproducible pipeline to run a quick Nerf reconstruction using `nerfstudio`, export geometry (pointcloud and mesh), and transfer semantic labels (from SAM2 masks) onto the reconstructed geometry.

### In this notebook we will:
1. Setup & environment checks
2. Paths and dataset split selection
3. Export images, intrinsics and `transforms.json`
4. Hardware-safe nerfstudio config generator
5. Run short training and monitor metrics
6. Export geometry (pointcloud + mesh)
7. SAM2 mask export and per-image masks
8. Semantic projection onto geometry
9. Visualization and comparison with MPS pointcloud (notebook 08)

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks` .

## 9.1 Setup & Environment Checks

This section verifies the Python environment and GPU availability. It also computes a recommended downscale factor for images based on available GPU memory (VRAM) to avoid out-of-memory during quick experiments.

In [2]:
import sys
import platform
import subprocess

# Torch / CUDA checks
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('Torch CUDA available:', torch.cuda.is_available())
    print('Torch CUDA version:', torch.version.cuda)
except Exception as e:
    print('PyTorch not available or import failed:', e)

# Try to query nvidia-smi for GPU names and memory (MB)
def query_nvidia_smi():
    try:
        out = subprocess.check_output([
            'nvidia-smi',
            '--query-gpu=name,memory.total',
            '--format=csv,noheader,nounits'
        ], encoding='utf-8')
        lines = [l.strip() for l in out.strip().splitlines() if l.strip()]
        infos = []
        for l in lines:
            name, mem = [x.strip() for x in l.split(',')]
            infos.append({'name': name, 'memory_mb': int(mem)})
        return infos
    except Exception:
        return []

gpu_infos = query_nvidia_smi()
if len(gpu_infos) == 0:
    # Fallback to torch if available
    try:
        if torch.cuda.is_available():
            prop = torch.cuda.get_device_properties(0)
            gpu_infos = [{'name': prop.name, 'memory_mb': int(prop.total_memory // 1024 // 1024)}]
    except Exception:
        pass

if gpu_infos:
    for i, g in enumerate(gpu_infos):
        print(f"GPU {i}: {g['name']} — {g['memory_mb']} MB VRAM")
else:
    print('No GPU detected or nvidia-smi not available.')


PyTorch: 2.11.0+cu130
Torch CUDA available: True
Torch CUDA version: 13.0
GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU — 6144 MB VRAM


## 9.2 Paths & Dataset Split Selection

This section defines the directory structure and selects which frames to use for training and evaluation. We will choose a sparse subset of frames (~10% as eval set) to balance training speed with reconstruction quality. The eval frames should be spatially diverse (not just temporally spread) to properly test generalization.

In [ ]:
import os
import numpy as np
import pandas as pd
from projectaria_tools.core import data_provider

# Define base directories
vrs_path = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'kettle_and_forklift_recording.vrs')
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
mps_dir = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Create output directory for nerfstudio dataset and training
nerfstudio_work_dir = os.path.join('..', 'data', 'outputs', 'nerfstudio', 'kettle_and_forklift')
os.makedirs(nerfstudio_work_dir, exist_ok=True)

# Subdirectories within nerfstudio work
dataset_dir = os.path.join(nerfstudio_work_dir, 'dataset')
images_dir = os.path.join(dataset_dir, 'images')
masks_export_dir = os.path.join(dataset_dir, 'masks')
training_dir = os.path.join(nerfstudio_work_dir, 'training')
exports_dir = os.path.join(nerfstudio_work_dir, 'exports')

# Create subdirectories
for d in [images_dir, masks_export_dir, training_dir, exports_dir]:
    os.makedirs(d, exist_ok=True)

print('Nerfstudio work directory:', nerfstudio_work_dir)
print('Dataset directory:', dataset_dir)
print('Training directory:', training_dir)
print('Exports directory:', exports_dir)

# Load VRS data provider to count available frames
provider = data_provider.create_vrs_data_provider(vrs_path)
stream_id = provider.get_stream_id_from_label('camera-rgb')
num_images = provider.get_num_data(stream_id)
print(f'\nTotal RGB frames available: {num_images}')

# Select frame indices for training and evaluation
# Strategy: use ~10% for eval (spatially diverse)
# For this demo, select frames evenly spaced and add some random frames for eval
all_frame_indices = np.arange(num_images, dtype=int)

# Heuristic: select every Nth frame for dense training, reserve sparse set for eval
stride = 5  # Use every 5th frame for training
train_frames_candidate = all_frame_indices[::stride]

# For eval, pick additional frames not in train set (e.g., frames at +2 offset within each stride window)
eval_frames = all_frame_indices[2::stride]  # Offset by 2 to ensure diversity

# Limit eval set to ~10% of the total
target_eval_count = max(1, int(num_images * 0.1))
if len(eval_frames) > target_eval_count:
    # Randomly sample from eval candidates
    np.random.seed(42)
    eval_frames = np.random.choice(eval_frames, size=target_eval_count, replace=False)
    eval_frames = sorted(eval_frames)

train_frames = sorted([f for f in train_frames_candidate if f not in eval_frames])

print(f'Selected {len(train_frames)} frames for training')
print(f'Selected {len(eval_frames)} frames for evaluation')
print(f'Eval frames: {eval_frames[:10]}... (showing first 10)')
print(f'Total frames in split: {len(train_frames) + len(eval_frames)}')